<a href="https://colab.research.google.com/github/FaarisIq/Persuasion-Analysis-Engine/blob/main/persuasion_analysis_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Persuasion Analysis Engine - Faaris Iqbal

In [ ]:
!python -m spacy download en_core_web_sm
!pip install praw pandas spacy vaderSentiment

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 10.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import re
import spacy
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np
from datetime import datetime
from convokit import Corpus, download
from scipy import stats


"""
Persuasion Analysis Engine - Faaris Iqbal

Analyzes persuasiveness in r/ChangeMyView using the Cornell ConvoKit
winning-args-corpus dataset with proper delta tracking.
"""


nlp = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

SCORING_WEIGHTS = {
    'top_comment_quality': 0.30,
    'discussion_depth': 0.20,
    'evidence_quality': 0.15,
    'engagement_quality': 0.15,
    'sophistication': 0.10,
    'emotion_balance': 0.10
}


def safe_get(data, key, default=0.0):
    if hasattr(data, 'get'):
        value = data.get(key, default)
    else:
        value = default
    if pd.isna(value) or value is None:
        return default
    try:
        return float(value) if isinstance(default, (int, float)) else value
    except (ValueError, TypeError):
        return default


def safe_str(value, default=''):
    if pd.isna(value) or value is None:
        return default
    return str(value)


def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'&gt;.*?\n', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:-]', '', text)
    return text.strip()


def spacy_sent_tokenize(text):
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    except:
        return [s.strip() for s in text.split('.') if s.strip()]


def analyze_source_credibility(text):
    if not text or not isinstance(text, str):
        return 0.0

    bare_urls = re.findall(r'https?://[^\s<>\[\])\]]+', text)
    markdown_urls = re.findall(r'\[(?:[^\]]*)\]\((https?://[^)]+)\)', text)
    urls = list(set(bare_urls + markdown_urls))

    if not urls:
        return 0.0

    credibility_score = 0.0

    high_cred_domains = [
        '.edu', '.gov', 'scholar.google', 'jstor', 'pubmed',
        'nature.com', 'science.org', 'cell.com', 'nejm.org',
        'stanford.edu', 'harvard.edu', 'mit.edu',
        'who.int', 'nih.gov', 'cdc.gov'
    ]

    medium_cred_domains = [
        'reuters.com', 'ap.org', 'bbc.com', 'npr.org',
        'economist.com', 'wsj.com', 'nytimes.com',
        'washingtonpost.com', 'theguardian.com',
        'wikipedia.org', 'britannica.com'
    ]

    low_cred_indicators = [
        'blog', 'wordpress', 'medium.com', 'reddit.com',
        'youtube.com', 'twitter.com', 'facebook.com',
        'tiktok.com', 'instagram.com'
    ]

    for url in urls:
        url_lower = url.lower()
        if any(domain in url_lower for domain in high_cred_domains):
            credibility_score += 1.0
        elif any(domain in url_lower for domain in medium_cred_domains):
            credibility_score += 0.7
        elif any(indicator in url_lower for indicator in low_cred_indicators):
            credibility_score += 0.2
        else:
            credibility_score += 0.4

    return min(credibility_score / len(urls), 1.0)


def detect_persuasive_features(text):
    if not text or not isinstance(text, str):
        return 0.0

    text_lower = text.lower()
    score = 0.0

    aggressive_words = [
        'stupid', 'idiotic', 'ridiculous', 'absurd', 'moronic',
        'dumb', 'idiot', 'fool', 'pathetic', 'nonsense',
        'bullshit', 'crap', 'garbage'
    ]
    aggressive_count = sum(1 for w in aggressive_words if w in text_lower)
    score -= min(aggressive_count * 0.1, 0.3)

    direct_quotes = len(re.findall(r'>\s*\S+', text))
    score += min(direct_quotes * 0.15, 0.25)

    specific_indicators = [
        r"for example", r"for instance", r"specifically",
        r"in particular", r"case in point", r"consider",
        r"take \w+", r"such as", r"like when"
    ]
    specific_count = sum(1 for p in specific_indicators if re.search(p, text_lower))
    score += min(specific_count * 0.1, 0.2)

    constructive = [
        r"perhaps consider", r"might want to", r"have you thought",
        r"one way to think", r"another angle", r"reframe",
        r"another perspective", r"consider that", r"think about it"
    ]
    constructive_count = sum(1 for p in constructive if re.search(p, text_lower))
    score += min(constructive_count * 0.12, 0.25)

    word_count = len(text.split())
    if 100 <= word_count <= 400:
        score += 0.20
    elif 50 <= word_count < 100 or 400 < word_count <= 800:
        score += 0.10
    elif word_count > 1000:
        score -= 0.05

    personal_markers = [
        r"i've experienced", r"in my experience", r"i worked",
        r"i lived", r"as someone who", r"i was", r"i remember",
        r"when i", r"my own"
    ]
    personal_count = sum(1 for p in personal_markers if re.search(p, text_lower))
    score += min(personal_count * 0.08, 0.15)

    hedging_patterns = [
        r"i think", r"perhaps", r"it seems", r"arguably",
        r"it could be", r"maybe", r"possibly", r"in my view"
    ]
    hedging_count = sum(1 for p in hedging_patterns if re.search(p, text_lower))
    score += min(hedging_count * 0.05, 0.15)

    concession_patterns = [
        r"you're right that", r"valid point", r"i agree that",
        r"fair point", r"you make a good point", r"i can see",
        r"i understand why", r"that's true", r"good question"
    ]
    concession_count = sum(1 for p in concession_patterns if re.search(p, text_lower))
    score += min(concession_count * 0.12, 0.25)

    certain_words = [
        r"obviously", r"clearly", r"definitely", r"absolutely",
        r"undoubtedly", r"without question", r"there is no doubt"
    ]
    certain_count = sum(1 for p in certain_words if re.search(p, text_lower))
    score -= min(certain_count * 0.05, 0.2)

    analogy_patterns = [
        r"it's like", r"imagine if", r"similar to", r"comparable to",
        r"think of it as", r"analogous to", r"the same as"
    ]
    analogy_count = sum(1 for p in analogy_patterns if re.search(p, text_lower))
    score += min(analogy_count * 0.1, 0.2)

    return min(max(score, 0.0), 1.0)


def detect_argument_sophistication(text):
    if not text:
        return 0.0

    text_lower = text.lower()

    counter_patterns = [
        r"however,", r"that said", r"on the other hand", r"although",
        r"but consider", r"while.*?true", r"granted,",
        r"i understand.*?but", r"you have a point.*?but"
    ]
    counter_count = sum(1 for p in counter_patterns if re.search(p, text_lower))
    counter_score = min(counter_count * 0.2, 1.0)

    logic_patterns = [
        r"because", r"therefore", r"thus", r"consequently",
        r"as a result", r"this leads to", r"it follows that",
        r"the reason", r"that's why"
    ]
    logic_count = sum(1 for p in logic_patterns if re.search(p, text_lower))
    logic_score = min(logic_count * 0.15, 1.0)

    evidence_patterns = [
        r"according to", r"research shows", r"studies indicate",
        r"data suggests", r"evidence shows"
    ]
    evidence_count = sum(1 for p in evidence_patterns if re.search(p, text_lower))
    evidence_score = min(evidence_count * 0.2, 1.0)

    total = (
        0.50 * counter_score +
        0.30 * logic_score +
        0.20 * evidence_score
    )

    return min(total, 1.0)


def get_emotion_scores(text):
    if not text or pd.isna(text):
        return {
            'vader_compound': 0.0, 'vader_pos': 0.0,
            'vader_neg': 0.0, 'vader_neu': 0.0,
            'persuasive_emotion': 0.0
        }

    vader_scores = analyzer.polarity_scores(str(text))

    persuasive_emotions = [
        'understand', 'realize', 'believe', 'feel', 'think',
        'important', 'significant', 'crucial', 'essential'
    ]

    emotional_intensity = sum(1 for word in persuasive_emotions if word in str(text).lower())
    emotional_score = min(emotional_intensity / 15, 0.5)

    return {
        'vader_compound': vader_scores['compound'],
        'vader_pos': vader_scores['pos'],
        'vader_neg': vader_scores['neg'],
        'vader_neu': vader_scores['neu'],
        'persuasive_emotion': emotional_score
    }


def calculate_emotion_balance(text):
    emotion = get_emotion_scores(text)
    compound = emotion['vader_compound']

    if 0.0 <= compound <= 0.3:
        balance_score = 1.0
    elif -0.2 <= compound < 0.0:
        balance_score = 0.7
    elif 0.3 < compound <= 0.6:
        balance_score = 0.6
    elif -0.5 <= compound < -0.2:
        balance_score = 0.4
    elif compound > 0.6:
        balance_score = 0.3
    else:
        balance_score = 0.2

    return 0.7 * balance_score + 0.3 * emotion['persuasive_emotion']


def score_comment_persuasion(comment_text, evidence_score=None):
    if not comment_text or pd.isna(comment_text):
        return 0.0

    comment_text = str(comment_text)

    word_count = len(comment_text.split())
    if word_count < 20:
        return 0.0

    persuasive_features = detect_persuasive_features(comment_text)
    sophistication = detect_argument_sophistication(comment_text)

    if evidence_score is None:
        evidence_score = analyze_source_credibility(comment_text)

    emotion_balance = calculate_emotion_balance(comment_text)

    total = (
        0.40 * persuasive_features +
        0.25 * sophistication +
        0.20 * evidence_score +
        0.15 * emotion_balance
    )

    return round(min(max(total, 0.0), 1.0), 3)


def load_cornell_data(sample_size=None):
    """
    Load Cornell ConvoKit winning-args-corpus and convert to DataFrame format.

    Args:
        sample_size: If specified, only process first N conversations (for testing)
    """
    print("Loading Cornell ConvoKit winning-args-corpus...")
    corpus = Corpus(filename=download("winning-args-corpus"))

    print(f"   Loaded {len(corpus.conversations)} conversations")
    print(f"   Loaded {len(corpus.utterances)} utterances")

    print("\nConverting to DataFrame format...")

    comments_data = []
    posts_data = []

    conv_count = 0

    for conv in corpus.iter_conversations():
        if sample_size and conv_count >= sample_size:
            break
        conv_count += 1

        # Get the conversation (post) info
        post_id = conv.id
        post_meta = dict(conv.meta) if conv.meta else {}

        # First utterance is usually the OP post
        utterances = list(conv.iter_utterances())
        if not utterances:
            continue

        # The OP utterance has the original post text
        op_utterance = None
        for utt in utterances:
            if utt.reply_to is None:  # Root utterance
                op_utterance = utt
                break

        if op_utterance is None:
            op_utterance = utterances[0]

        # Build post record
        post_record = {
            'post_id': post_id,
            'title': post_meta.get('title', ''),
            'author': op_utterance.speaker.id,
            'post_text': str(op_utterance.text),
            'original_post_text': str(op_utterance.text),
            'post_score': post_meta.get('score', 0),
            'num_comments': len(utterances) - 1,
            'created_utc': op_utterance.timestamp,
        }

        # Count deltas in this conversation
        delta_count = sum(1 for utt in utterances if utt.meta.get('success', 0) == 1)
        post_record['delta_count'] = delta_count
        post_record['has_deltas'] = delta_count > 0

        posts_data.append(post_record)

        # Build comment records (skip the OP post)
        for utt in utterances:
            if utt.id == op_utterance.id:
                continue

            comment_record = {
                'post_id': post_id,
                'comment_id': utt.id,
                'parent_id': utt.reply_to,
                'author': utt.speaker.id,
                'comment_text': str(utt.text),
                'comment_score': utt.meta.get('score', 0),
                'created_utc': utt.timestamp,
                'is_root_comment': utt.reply_to == op_utterance.id,
                'won_delta': utt.meta.get('success', 0) == 1,
            }

            comments_data.append(comment_record)

        if conv_count % 100 == 0:
            print(f"   Processed {conv_count} conversations, {len(comments_data)} comments so far")

    posts_df = pd.DataFrame(posts_data)
    comments_df = pd.DataFrame(comments_data)

    print(f"\nFinal dataset:")
    print(f"   Posts: {len(posts_df)}")
    print(f"   Comments: {len(comments_df)}")
    print(f"   Delta winners: {comments_df['won_delta'].sum()}")

    return posts_df, comments_df


def analyze_comments(comments_df):
    print(f"\nAnalyzing {len(comments_df)} comments...")

    enhanced_comments = []
    skipped_count = 0

    for idx, comment in comments_df.iterrows():
        comment_text = safe_str(comment.get('comment_text', ''))

        if not comment_text or len(comment_text.split()) < 20:
            skipped_count += 1
            continue

        evidence = analyze_source_credibility(comment_text)
        persuasion_score = score_comment_persuasion(comment_text, evidence_score=evidence)
        persuasive_features = detect_persuasive_features(comment_text)
        sophistication = detect_argument_sophistication(comment_text)
        emotion_balance = calculate_emotion_balance(comment_text)
        emotion = get_emotion_scores(comment_text)

        enhanced_comment = comment.to_dict()
        enhanced_comment.update({
            'persuasion_score': persuasion_score,
            'persuasive_features_score': persuasive_features,
            'sophistication_score': sophistication,
            'evidence_quality_score': evidence,
            'emotion_balance_score': emotion_balance,
            'vader_compound': emotion['vader_compound'],
            'vader_positive': emotion['vader_pos'],
            'vader_negative': emotion['vader_neg'],
            'vader_neutral': emotion['vader_neu'],
            'is_substantial': len(comment_text.split()) >= 20,
            'is_high_quality': persuasion_score >= 0.2,
        })

        enhanced_comments.append(enhanced_comment)

        if (idx + 1) % 5000 == 0:
            print(f"   Processed {idx + 1}/{len(comments_df)} comments (skipped {skipped_count})")

    print(f"   Completed: {len(enhanced_comments)} comments analyzed, {skipped_count} skipped")
    return pd.DataFrame(enhanced_comments)


def save_results(posts_df, comments_df):
    posts_df.to_csv('cornell_posts_results.csv', index=False)
    print(f"Posts saved to 'cornell_posts_results.csv'")

    comments_df.to_csv('cornell_comments_results.csv', index=False)
    print(f"Comments saved to 'cornell_comments_results.csv'")


def statistical_analysis(comments_df):
    """
    Run statistical tests comparing delta winners vs non-delta comments.
    Uses REAL delta labels from Cornell data.
    """
    print("\n" + "=" * 60)
    print("Statistical Analysis: Delta Winners vs Non-Delta Comments")
    print("=" * 60)

    delta_winners = comments_df[comments_df['won_delta'] == True]
    non_delta = comments_df[comments_df['won_delta'] == False]

    print(f"\nSample sizes:")
    print(f"   Delta-winning comments: {len(delta_winners)}")
    print(f"   Non-delta comments: {len(non_delta)}")

    if len(delta_winners) == 0:
        print("Warning: No delta-winning comments identified.")
        return None

    metrics_to_test = [
        'persuasion_score',
        'persuasive_features_score',
        'sophistication_score',
        'evidence_quality_score',
        'emotion_balance_score',
        'vader_compound'
    ]

    results = []

    print(f"\nT-Test Results (Delta vs Non-Delta):")
    print("-" * 80)
    print(f"{'Metric':<30} {'Delta':>8} {'Non-Delta':>10} {'t-stat':>8} {'p-value':>10} {'Cohen d':>8}")
    print("-" * 80)

    for metric in metrics_to_test:
        if metric not in comments_df.columns:
            continue

        delta_vals = delta_winners[metric].dropna()
        non_delta_vals = non_delta[metric].dropna()

        if len(delta_vals) == 0 or len(non_delta_vals) == 0:
            continue

        t_stat, p_value = stats.ttest_ind(delta_vals, non_delta_vals, equal_var=False)

        pooled_std = np.sqrt(
            ((len(delta_vals) - 1) * delta_vals.var() +
             (len(non_delta_vals) - 1) * non_delta_vals.var()) /
            (len(delta_vals) + len(non_delta_vals) - 2)
        )
        cohens_d = (delta_vals.mean() - non_delta_vals.mean()) / pooled_std if pooled_std > 0 else 0

        results.append({
            'metric': metric,
            'delta_mean': delta_vals.mean(),
            'non_delta_mean': non_delta_vals.mean(),
            't_statistic': t_stat,
            'p_value': p_value,
            'cohens_d': cohens_d,
            'n_delta': len(delta_vals),
            'n_non_delta': len(non_delta_vals)
        })

        if p_value < 0.001:
            p_str = "<0.001"
        else:
            p_str = f"{p_value:.4f}"

        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"

        print(f"{metric:<30} {delta_vals.mean():>8.3f} {non_delta_vals.mean():>10.3f} "
              f"{t_stat:>8.3f} {p_str:>10} {cohens_d:>8.3f} {sig}")

    print("\n" + "=" * 60)
    print("Effect Size Interpretation (Cohen's d):")
    print("   |d| < 0.2: negligible")
    print("   |d| < 0.5: small")
    print("   |d| < 0.8: medium")
    print("   |d| >= 0.8: large")
    print("\nSignificance Codes: *** p<0.001, ** p<0.01, * p<0.05, ns: not significant")

    return pd.DataFrame(results)


def baseline_comparisons(comments_df):
    """Compare framework against simple baselines."""
    print("\n" + "=" * 60)
    print("Baseline Comparisons")
    print("=" * 60)

    from sklearn.metrics import roc_auc_score

    comments_df['word_count'] = comments_df['comment_text'].apply(
        lambda x: len(str(x).split()) if not pd.isna(x) else 0
    )

    baselines = {
        'Word count alone': 'word_count',
        'Sentiment alone (VADER compound)': 'vader_compound',
        'Persuasive features only': 'persuasive_features_score',
        'Sophistication only': 'sophistication_score',
        'Our composite framework': 'persuasion_score'
    }

    print(f"\n{'Method':<40} {'AUC':>8}")
    print("-" * 50)

    results = []

    for name, column in baselines.items():
        try:
            if column in comments_df.columns:
                scores = comments_df[column]
                valid_mask = ~scores.isna()

                if valid_mask.sum() > 0 and comments_df.loc[valid_mask, 'won_delta'].nunique() > 1:
                    auc = roc_auc_score(
                        comments_df.loc[valid_mask, 'won_delta'].astype(int),
                        scores[valid_mask]
                    )
                    results.append({'method': name, 'auc': auc})
                    print(f"{name:<40} {auc:>8.3f}")
        except Exception as e:
            print(f"{name:<40} Error: {str(e)[:30]}")

    return pd.DataFrame(results)


def cross_validation_analysis(comments_df, n_folds=10):
    """Perform k-fold cross-validation."""
    print("\n" + "=" * 60)
    print(f"{n_folds}-Fold Cross-Validation")
    print("=" * 60)

    from sklearn.model_selection import KFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler

    feature_cols = [
        'persuasive_features_score',
        'sophistication_score',
        'evidence_quality_score',
        'emotion_balance_score'
    ]

    df_clean = comments_df.dropna(subset=feature_cols + ['won_delta'])

    if df_clean['won_delta'].sum() < 5:
        print("Warning: Too few delta winners for reliable cross-validation.")
        return None

    X = df_clean[feature_cols].values
    y = df_clean['won_delta'].astype(int).values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    auc_scores = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_scaled)):
        model = LogisticRegression(class_weight='balanced', max_iter=1000)
        model.fit(X_scaled[train_idx], y[train_idx])

        y_pred_proba = model.predict_proba(X_scaled[test_idx])[:, 1]

        if len(np.unique(y[test_idx])) > 1:
            auc = roc_auc_score(y[test_idx], y_pred_proba)
            auc_scores.append(auc)

    if auc_scores:
        mean_auc = np.mean(auc_scores)
        std_auc = np.std(auc_scores)
        print(f"\nMean AUC: {mean_auc:.3f} ± {std_auc:.3f}")
        print(f"Min AUC: {min(auc_scores):.3f}")
        print(f"Max AUC: {max(auc_scores):.3f}")
        print(f"Folds with valid AUC: {len(auc_scores)}/{n_folds}")

        return {
            'mean_auc': mean_auc,
            'std_auc': std_auc,
            'all_scores': auc_scores
        }

    return None


def display_results(comments_df):
    print("\n" + "=" * 60)
    print("Persuasion Analysis Results - Cornell Data")
    print("=" * 60)

    print(f"\nDataset:")
    print(f"   Total comments analyzed: {len(comments_df):,}")
    print(f"   Substantial comments: {comments_df['is_substantial'].sum():,}")
    print(f"   High-quality comments: {comments_df['is_high_quality'].sum():,}")
    print(f"   Delta-winning comments: {comments_df['won_delta'].sum():,}")

    print(f"\nPersuasion Scores:")
    print(f"   Average: {comments_df['persuasion_score'].mean():.3f}")
    print(f"   Highest: {comments_df['persuasion_score'].max():.3f}")
    print(f"   Lowest: {comments_df['persuasion_score'].min():.3f}")
    print(f"   Std dev: {comments_df['persuasion_score'].std():.3f}")

    # Compare delta winners vs non-delta
    delta_comments = comments_df[comments_df['won_delta'] == True]
    non_delta_comments = comments_df[comments_df['won_delta'] == False]

    print(f"\nDelta Winners vs Non-Delta Comparison:")
    print(f"   Delta winner avg score: {delta_comments['persuasion_score'].mean():.3f}")
    print(f"   Non-delta avg score: {non_delta_comments['persuasion_score'].mean():.3f}")

    diff = delta_comments['persuasion_score'].mean() - non_delta_comments['persuasion_score'].mean()
    pct_diff = (diff / non_delta_comments['persuasion_score'].mean()) * 100
    print(f"   Difference: +{diff:.3f} ({pct_diff:.1f}% increase)")


if __name__ == "__main__":
    print("CMV Persuasion Analysis - Cornell ConvoKit Data")
    print("-" * 50)

    # Load Cornell data (set sample_size=100 for testing, None for full dataset)
    posts_df, comments_df = load_cornell_data(sample_size=None)

    if posts_df is None or comments_df is None:
        print("\nFailed to load data.")
        exit(1)

    # Analyze all comments
    analyzed_comments = analyze_comments(comments_df)

    # Save raw analyzed data
    save_results(posts_df, analyzed_comments)

    # Display basic results
    display_results(analyzed_comments)

    # Statistical analysis
    stat_results = statistical_analysis(analyzed_comments)
    if stat_results is not None:
        stat_results.to_csv('cornell_statistical_results.csv', index=False)
        print("\nStatistical results saved to 'cornell_statistical_results.csv'")

    # Baseline comparisons
    baseline_results = baseline_comparisons(analyzed_comments)
    if baseline_results is not None:
        baseline_results.to_csv('cornell_baseline_results.csv', index=False)
        print("Baseline results saved to 'cornell_baseline_results.csv'")

    # Cross-validation
    cv_results = cross_validation_analysis(analyzed_comments)

    print("\nAnalysis complete.")

CMV Persuasion Analysis - Cornell ConvoKit Data
--------------------------------------------------
Loading Cornell ConvoKit winning-args-corpus...
   Loaded 3051 conversations
   Loaded 293297 utterances

Converting to DataFrame format...
   Processed 100 conversations, 10486 comments so far
   Processed 200 conversations, 20220 comments so far
   Processed 300 conversations, 30261 comments so far
   Processed 400 conversations, 38839 comments so far
   Processed 500 conversations, 48557 comments so far
   Processed 600 conversations, 58087 comments so far
   Processed 700 conversations, 68026 comments so far
   Processed 800 conversations, 77113 comments so far
   Processed 900 conversations, 87989 comments so far
   Processed 1000 conversations, 97394 comments so far
   Processed 1100 conversations, 107445 comments so far
   Processed 1200 conversations, 115780 comments so far
   Processed 1300 conversations, 124771 comments so far
   Processed 1400 conversations, 134277 comments so 